# 1 · Panel construction

From the raw PitchBook tables to the panel: one row per year of life of every
company, from its founding year to the last year anything is known about it.

Each step below is one call to a function of `src/panel/`, with what it does and
why above it, and what came out of it printed underneath. The command line runs
the same functions without the narration:

```bash
python scripts/pipeline.py panel --both          # both panels
python scripts/pipeline.py panel --example       # on the synthetic extraction
```

## The two configurations

Every attribute of a person, an investor or a competitor can be read in two ways:
as it was **in the year of the row**, or as it was **declared at extraction
time**. The first is what a model could have known at the time; the second is the
look-ahead the paper measures. The two panels are the same pipeline with this one
switch flipped, so that everything else is held constant.

| `TIMED` | file | dataset it feeds |
|---|---|---|
| `True` | `panel_timed.parquet`, `panel_timed.csv.gz` | `dataset_controlled` |
| `False` | `panel_snapshot.parquet`, `panel_snapshot.csv.gz` | `dataset_leakboth` |

## Without the PitchBook data

The extraction cannot be redistributed. Set `EXAMPLE = True` below and everything
runs on `data/example/pitchbook/`: fourteen invented tables, eight companies, each
built so that one branch of the pipeline fires. It takes seconds, and the numbers
in the printouts are small enough to follow by hand.

In [ ]:
import polars as pl
import yaml

from src.panel import checks, companies, competitors, deals, people, stages, target, team
from src.panel.config import PanelConfig, PanelRules
from src.panel.pipeline import panel_name

# EXAMPLE reads the synthetic extraction that ships with the repository; TIMED
# chooses which of the two panels this run builds.
EXAMPLE = False
TIMED = True

CONFIG = yaml.safe_load(open("config/config.yaml"))
RULES = PanelRules.from_config(CONFIG)
MIN_FOUNDING_YEAR = int(CONFIG["first_year"])

cfg = (
    PanelConfig(raw_dir="data/example/pitchbook", interim_dir="data/example/interim")
    if EXAMPLE
    else PanelConfig()
)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(40)

print("extraction :", cfg.raw_dir)
print("outputs    :", cfg.interim_dir)
print("attributes :", "of the row's own year" if TIMED else "as declared at extraction time")
print("sample     : companies founded from", MIN_FOUNDING_YEAR, "on")

---
## Phase 1 - the skeleton

This is where the shape of the panel is decided. Eleven columns of `Company.csv`
are read; four of them, together with the fiscal period, produce no panel column
at all and only serve to find the last year with data.

**The dates.** Everything is read as text and cast deliberately: an identifier
that looks numeric would break a join silently if it became a number. The
missing-value tokens are applied first, because what counts as missing decides
every step that follows. The fiscal period arrives as `TTM 2Q2019` and becomes the
date the quarter closes on.

In [ ]:
all_companies = companies.read_companies(cfg)
print(f"{all_companies.height:,} companies x {all_companies.width} columns")
all_companies.select("CompanyID", "YearFounded", "HQCountry", "OwnershipStatusDate", "FiscalDate").head(3)

**The window of life.** The last known year is the most recent of six dates: it is
the last year the source knows anything about that company, and the panel of that
company ends there, because after it not even the outcome could be read. A company
without a founding year, or with no date at all, does not enter the panel.

The same window is computed once and used twice, here and by the competitors of
phase 7, so the two cannot disagree.

In [ ]:
life = companies.company_life(all_companies)
print(f"companies with a usable window: {life.height:,} of {all_companies.height:,}")
life.head(3)

**The skeleton.** One row per year, from the founding year to the last known one.
The upper bound is the later of the two, because a few records carry dates that
precede the founding year (re-registrations, mostly) and without the bound they
would produce years before the company existed.

The ownership status is one row per company, that is the state as of the
extraction, and its date says when it was reached: it is therefore attached to
that year alone and stays empty on every other row. The growth-stage cascade of
phase 5 reads it.

In [ ]:
skeleton = companies.build_skeleton(life, min_founding_year=MIN_FOUNDING_YEAR)
skeleton = companies.attach_ownership_status(skeleton, all_companies)
print(f"skeleton: {skeleton.height:,} company-years, {skeleton['CompanyID'].n_unique():,} companies")
print(f"rows with an ownership status: {skeleton['OwnershipStatus'].is_not_null().sum():,}")
skeleton.head(4)

**The registry.** The company-level attributes the later phases need: two of them
are panel columns that never vary by year and are attached at the end, the other
two serve the repair of the deal dates.

In [ ]:
registry = companies.company_registry(all_companies, min_founding_year=MIN_FOUNDING_YEAR)
del all_companies
print(f"registry: {registry.height:,} companies x {registry.width} columns")

---
## Phase 2 - one row per (company, person)

The board table has one row per **appointment**, so the same person appears more
than once in the same company when they held more than one role, or when the same
role was recorded twice. Team variables are per person, not per role, so the rows
are merged, and the two situations are merged by different rules, because in one
the duplication is noise and in the other it is information.

**The merge.** For the *same appointment* recorded twice the more reliable version
wins: "still in office" beats a leaving date, a date beats a missing value, and
when both were updated on the same day the wider interval wins. For *different
roles* of the same person the periods are merged by **union**, and the titles are
concatenated so that a founder stays a founder.

In [ ]:
board = people.merge_appointments(people.read_board(cfg, skeleton))
print(f"one row per pair: {board.height:,}")
board.head(3)

**The chief-executive roles**, with the years each one covers. They are put aside
now because further on the dates become ages of the company and the title is
dropped. Mind what these dates mean, because it decides what phase 5 can do with
them: the start is when the person joined the company, not when they became chief
executive.

In [ ]:
years = people.company_years(skeleton)
ceo_roles = people.ceo_roles(board, years, RULES)
print(f"chief-executive roles: {ceo_roles.height:,}"
      f"  (founder and chief executive: {ceo_roles['is_founder'].sum():,},"
      f" with a declared start: {ceo_roles['sv'].is_not_null().sum():,})")

**The attributes of a person**, and who is a founder. A degree in the name (Ph.D,
JD, MD) carries no date and counts in every year. A founder is recognised from
the title of the appointment or from the position level, whichever declares it;
the pattern covers the misspellings the source carries without catching
"Foundation", and a *founder's associate* is an assistant and not a founder.

In [ ]:
pairs = people.person_attributes(board, cfg, RULES)
print(f"founders: {pairs['IsFounder'].sum():,} of {pairs.height:,} pairs"
      f"   (null: {pairs['IsFounder'].null_count()})")
print(f"no gender declared: {pairs['Gender'].is_null().sum():,}")

**From when to when.** Three rules, and they decide every team column.

*Arrival.* A founder starts at year zero, even where a later date exists: that is
what being a founder means. A date before the founding year is moved to it. Someone
who is **not** a founder and has no start date is not counted at all: starting them
at the founding year would put people who arrived years later in the team of the
early years, and those people are more numerous precisely in the companies that go
on to grow. The row stays, because the chief-executive step reads it.

*Departure.* A missing end date becomes the company's last year, whatever the
declared status: counting a person one year too long costs less than losing them.

*The cut.* No window goes past the company's last year, because beyond it nothing
is known about the company and those years could not carry a target.

In [ ]:
pairs = people.presence_window(pairs, years)
print(f"pairs: {pairs.height:,} x {pairs.width} columns")
print(f"not counted in the team (no arrival year): {pairs['DeltaStart'].null_count():,}")
pairs.head(3)

**Experience, year by year.** The source exposes eight counters of roles, but they
are totals at extraction time and cannot say how many roles a person had in 2012.
Each one is rebuilt from a detail table with one row per role, and every role
becomes an event with a year: the experience at year Y is the number of roles
started by Y, finished or not.

The year of a role is chosen in cascade: the declared date; the founding year of
the entity the role is held in, which is not the real date but a lower bound; the
person's first known year; and year zero for someone with no dated role at all.

In [ ]:
experience = people.experience_events(cfg, board)
print(f"experience: {experience.height:,} rows for {experience['PersonID'].n_unique():,} people")
experience.head(3)

**The check.** Every counter of `Person.csv` against the rows of its own detail
table, and the sum per group against the last row of the rebuilt table, which by
construction holds every role of any year. The second comparison raises if it
fails, because it would be a defect of the step above; whatever difference the
first one shows is an inconsistency of the source, and the detail table is the
more complete of the two.

In [ ]:
people.check_experience_counts(cfg, experience)

**Education, year by year.** For a given year only the degrees earned by then
count, otherwise a master's taken in 2018 would raise the education of the row of
2010. A degree with no graduating year counts in every year, one dated in the
future in none of the panel's. Only the years in which something changed are
stored; the join by year downstream picks the most recent one that is not later
than the row.

The degree levels and the fields of study are the rules in `config.yaml`, applied
in order: the first pattern that matches wins, which is why `MD` never reaches
`Master's`.

In [ ]:
studies = people.classify_studies(cfg, RULES)
education = people.education_by_year(studies, board, RULES)
changing = people.check_education(studies, education, board, RULES)
print(f"education: {education.height:,} rows for {education['PersonID'].n_unique():,} people")
print(f"people whose education changes over time: {changing:,}")
education.head(3)

---
## Phase 3 - the team columns

Every person is expanded over the years of their window, the attributes of that
year are attached, and the rows are collapsed back to one per (company, year).

In [ ]:
del board
expanded = team.expand_team(pairs, min_founding_year=MIN_FOUNDING_YEAR)
print(f"(company, year, person) rows: {expanded.height:,}")

**The attributes of the year.** Two as-of joins take, for each row, the person's
most recent row that is not later than the year of the panel row. With `TIMED`
off they take the person's last row instead, which is the value declared at
extraction time.

In [ ]:
expanded = team.attach_person_attributes(expanded, experience, education, RULES, timed=TIMED)
print(f"a degree by that year on {expanded['Highest_Degree'].is_not_null().mean():.1%} of the rows")

**The experience index** is the mean of the three counts, compressed by a
logarithm and standardised. With `TIMED` on, mean and standard deviation are
computed **per year over an expanding window**: for the row of a given year only
the (person, year) pairs of that year or earlier take part. Otherwise the value of
a row of 2005 would depend on the rows of 2020, and the early years (when few
people are documented) would be scored on the scale of two decades later.

In [ ]:
parameters = team.experience_parameters(expanded, timed=TIMED)
expanded = team.experience_index(expanded, parameters, RULES, timed=TIMED)
print(f"experience index: mean {expanded['WorkExperienceIndex'].mean():.4f}")
parameters.head(3)

**The fifteen aggregates.** How many people, the share of women, the eight flags
on fields of study, the mean graduation year, the mean degree, the institutes, the
mean experience index and how many founders.

The denominator of the share of women is the people whose gender is known: keeping
the others in it would lower the share precisely in the companies documented
worst, and where nobody's gender is known the column stays empty.

The panel **is** the skeleton, and the team is attached to its years with a left
join: a company-year covered only by the team would fall past the company's last
year, where there is no target. After the cut of phase 2 that cannot happen, and
the function checks it rather than trusting it.

In [ ]:
panel = team.join_skeleton(skeleton, team.aggregate_team(expanded, RULES))
del expanded, skeleton
print(f"panel: {panel.height:,} rows x {panel.width} columns")
print(f"rows with team data: {panel['Total_People'].is_not_null().sum():,}")

---
## Phase 4 - the funding rounds

The rounds decide the growth stage, and the growth stage is the target, so this
phase carries more decisions than any other.

**The investors**, collapsed into seven categories. Two quantities describe each
one (how many investments it has made and the median size of the rounds it takes
part in) and together they say how large and how active whoever puts the money in
is. With `TIMED` on they are rebuilt year by year from the dated rounds; the price
is that the extraction only holds the rounds of the companies in the sample, so
the large funds come out smaller than they are.

Then the rounds are described by the investors that entered them. Every aggregate
is conditional on "is there at least one new investor?", and that question has
three answers: yes, no, and *we cannot tell* when a status is missing, in which
case the aggregate is null rather than a no.

In [ ]:
participations = deals.investor_participations(cfg, RULES, timed=TIMED)
by_deal = deals.aggregate_by_deal(participations, RULES)
del participations
print(f"rounds with at least one investor on record: {by_deal.height:,}")

**The dates that are missing.** Some rounds carry none, and four steps give them
one: a bankruptcy takes the date of the ownership change, so does an
acquisition, a first round of an initial kind goes to the founding year, and the
rounds between two dated ones are spread evenly over the gap. The bounds of the
last step are always real rounds, so a round with no dated round beside it keeps no
date, and will leave the panel, which is why the companies that lose rounds are
recorded.

In [ ]:
rounds = deals.read_deals(cfg, registry)
print(f"raw rounds: {rounds.height:,}   without a date: {rounds['DealDate'].is_null().sum():,}")
rounds = deals.repair_deal_dates(rounds, RULES, min_founding_year=MIN_FOUNDING_YEAR)
print(f"in the sample: {rounds.height:,}   still without a date: {rounds['DealDate'].is_null().sum():,}")

**Into a year of the panel.** A round dated before the founding year is moved to
it, because the panel starts at age zero and there is no earlier row to land on. A
round with no year keeps none, joins nothing, and disappears: the list of who loses
rounds is written beside the panel for the robustness check, since the bias those
rounds introduce goes one way only, a round nobody dated cannot raise a stage.

In [ ]:
rounds = deals.place_deals_in_years(rounds, by_deal)
lost = deals.companies_losing_rounds(rounds, RULES)
del by_deal
print(f"rounds leaving the panel: {rounds['Year_Delta'].is_null().sum():,}")
print(f"companies affected: {lost.height:,}, of which lose every round: {lost['loses_all'].sum():,}")
lost.head(3)

**The flags, and the aggregation by company-year.** The type of a round becomes
fourteen flags, nine of which are not panel columns but the cascade that produces
the growth stage. A flag is on when at least one round of that year turns it on;
the accelerator and angel ones are an OR with the category of the investors, so
that a round from an accelerator counts even when the type does not say so.

The capital raised is the sum of the amounts actually declared, and an unknown
amount is added as a zero, which is why the share of rounds with an undisclosed
amount travels beside it.

In [ ]:
rounds = deals.add_deal_flags(rounds, RULES)
by_company_year = deals.aggregate_by_company_year(rounds, RULES)
panel = deals.attach_deals(panel, by_company_year)
del rounds, by_company_year
print(f"panel: {panel.height:,} rows x {panel.width} columns")
print(f"company-years with no round at all: {(panel['TR_D'] == 1).sum():,}")

---
## Phase 5 - stages, cumulative totals and the chief executive

**The flags become cumulative**: once on, on for every later year. That is how
"closed a seed round this year" becomes "has already closed a seed round", and it
is what makes the stages monotone. Most company-years hold no round at all, so
without it the stage would be empty on most rows and the target would not stand up.

**The stage** comes from a cascade that short-circuits, so the order of its
branches is part of the definition. The first three are terminal, and in them the
ownership status competes with the flags, because there are companies that closed
and no round records it: without the status they would sit in the panel as if they
were alive.

In [ ]:
panel = stages.cumulate_flags(panel, RULES)
panel = stages.growth_stage(panel)
panel["GrowthStage"].value_counts(sort=True)

**The cumulative totals.** The number of rounds and of investors accumulates; the
two investor measures become cumulative means weighted by how many new investors
entered in each year, so that a round with ten investors counts ten times one with
a single investor. A year with no round raised nothing, and has no undisclosed
amounts either.

In [ ]:
panel = stages.cumulative_totals(panel)
print(f"rounds, maximum: {panel['N_Deal'].max()}   investors, maximum: {panel['TotalInvestors'].max()}")

**Who the chief executive is.** The rounds declare one, so the name is known only
in the years that hold a round; it is carried forward. That leaves two problems.

*Before the first round there is no chief executive at all*, and those are exactly
the early years the models read their features from. The gap is filled from the
board, but only where the attribution can be verified: a single such role covers
the year, the title is a founder's as well, the start date is declared, it is the
same person the first round confirms, and nobody else holds a chief-executive role
that started earlier.

*When a new chief executive arrives between two rounds*, carrying the previous one
forward would attribute the wrong person, so a role that starts with a declared
date after the last declaration of the rounds wins.

In [ ]:
panel, filled, corrected = stages.resolve_ceo(panel, ceo_roles)
del ceo_roles
print(f"rows filled before the first round: {filled:,}")
print(f"attributions corrected: {corrected:,}")

**The attributes of the chief executive**: gender, experience index and highest
degree, read for the year of the row and with the same parameters as the team
index, so that the two live on one scale. They stay null where the chief executive
is not on the board of that same company.

Then country and sector, which never varied by year, and the age of the company.

In [ ]:
panel = stages.ceo_attributes(panel, pairs, experience, education, parameters, RULES, timed=TIMED)
panel = stages.attach_registry(panel, registry)
del pairs, experience, education, parameters, registry
print(f"rows with the chief executive's attributes: {panel['Gender_CEO'].is_not_null().sum():,}")
print(f"panel: {panel.height:,} rows x {panel.width} columns")

---
## Phase 6 - the groups, the future stage, the truncation

The seven stages collapse into four. Then the stage each company moves to next is
computed, **before** the truncation, and that order is a constraint: computing it
after would make an exit unreachable as a future stage, which is exactly what the
column is for.

The truncation cuts each company at its exit, the year of the exit included.
Dropping that year too is the condition for the target to be correct: the outcome
survives in the future stage, and the last row left says "about to leave, in N
years". Keeping it would move the year the target is read at onto that very row,
where the future stage points past the exit.

In [ ]:
panel = target.group_stages(panel, RULES)
panel = target.next_stage(panel)
before = panel.height
panel = target.truncate_at_exit(panel, RULES)
print(f"rows: {before:,} -> {panel.height:,}   companies: {panel['CompanyID'].n_unique():,}")
panel["GrowthNextStageGroup"].value_counts(sort=True)

---
## Phase 7 - the competitors, and the final shape

Three columns (how many competitors, how many of them in the same country, the
mean similarity) plus the number of similar companies that mean is computed over.

With `TIMED` on only the competitors *alive in that year* are counted, and a pair
survives only if the life window of the other company is known, which can be read
from the company table alone: most of the declared pairs do not survive the filter. With
`TIMED` off every pair on record counts, in every year, which is the baseline with
the look-ahead.

The heaviest assumption of the phase is that the last year with data stands for
"still alive": a company covered well comes out alive for longer than one covered
badly, so the count over-weights the large and the well-documented.

In [ ]:
similar, competitor_pairs = competitors.similar_pairs(cfg, panel, life, timed=TIMED)
print(f"pairs used: similar {similar.height:,}   competitors {competitor_pairs.height:,}")
competitor_stats, similarity_stats = competitors.competitor_columns(
    panel, similar, competitor_pairs, timed=TIMED
)
del similar, competitor_pairs, life
print(f"company-years with at least one competitor: {competitor_stats.height:,}")

**The final shape.** The competitor columns are grafted on, the placeholder in the
future stage becomes the current group, and the companies are **renumbered**,
the last operation of the pipeline, after which nothing can be joined back to the
source. The columns are then put in the order `config.yaml` declares.

In [ ]:
final = competitors.finalize(panel, competitor_stats, similarity_stats, RULES)
del panel, competitor_stats, similarity_stats
print(f"panel: {final.height:,} rows x {final.width} columns")
final.head(5)

---
## Checks, and writing the panel

The checks are structural: they hold of any extraction, because they are the rules
the pipeline enforces. No year precedes a founding year, no company-year appears
twice, the columns are the declared schema, nothing survives past an exit, and the
cumulative columns never go backwards. One of those failing is a bug, so it raises.

The same functions run in `scripts/pipeline.py` and in the test suite, so a check
cannot mean one thing here and another there.

In [ ]:
checks.verify(final, RULES)

In [ ]:
# The same files build_panel writes: the name is the one notebook 2 and the command
# line read, and the companies that lose rounds go beside the panel.
name = panel_name(timed=TIMED)
final.write_parquet(cfg.interim(f"{name}.parquet"))
final.write_csv(cfg.interim(f"{name}.csv.gz"), compression="gzip")
lost.write_parquet(cfg.interim("companies_losing_rounds.parquet"))
print("written:", cfg.interim(f"{name}.csv.gz"))

---
## Next

Run this notebook twice, once with `TIMED = True` and once with `TIMED = False`, to
get the two panels the comparison needs. Then continue with
**`2_dataset_construction.ipynb`**, which turns them into the two datasets the
models read.